# CSP vs. statsforecast conformal vs. model-native intervals — AirPassengers

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

This notebook reproduces the classic **AirPassengers** example from
*Mastering Modern Time-Series Forecasting* (Ch. 4, the seasonal `AutoARIMA` winner)
and adds **Conformal Seasonal Pools (CSP)** — the training-free probabilistic
forecaster from V. Manokhin (2026), [arXiv:2605.03789](https://arxiv.org/abs/2605.03789).

We forecast the last **24 months** and compare **three** kinds of 95% prediction interval
on the held-out window:

| Interval | Source | Training |
|---|---|---|
| **Model-native** | `AutoARIMA` analytical PI (`level=[95]`) | fits SARIMA |
| **statsforecast conformal** | same `AutoARIMA` wrapped in `ConformalIntervals` | fits SARIMA + split-conformal residuals |
| **CSP** | `csp_forecaster.CSPModel` | **none** (training-free) |

Metrics: empirical **coverage** @95%, mean **interval width** (sharpness), and the
**Winkler / interval score** (a proper score that rewards narrow intervals *and* penalises misses).

In [ ]:
# --- install (Colab) ---
%pip -q install statsforecast utilsforecast csp-forecaster matplotlib pandas

In [ ]:
import warnings; warnings.simplefilter("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA
from statsforecast.utils import ConformalIntervals
from csp_forecaster import CSPModel
np.random.seed(0)

## 1. Load AirPassengers and build the statsforecast long frame

In [ ]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
raw = pd.read_csv(url)
raw.columns = ["ds", "y"]
raw["ds"] = pd.to_datetime(raw["ds"])              # monthly, 1949-01 .. 1960-12 (144 pts)
raw["unique_id"] = "AirPassengers"
df = raw[["unique_id", "ds", "y"]]

H = 24                                             # forecast horizon (months)
M = 12                                             # seasonal period
train = df.iloc[:-H].copy()
test  = df.iloc[-H:].copy()
y_true = test["y"].to_numpy(dtype=float)
print(f"train={len(train)} months, test={len(test)} months, season={M}")
train.tail(3)

## 2. The book's best model: seasonal `AutoARIMA`
`AutoARIMA(season_length=12)` auto-selects a SARIMA — the strongest classical model for
AirPassengers in Ch. 4. We read two interval flavours off the **same** model.

In [ ]:
# (a) MODEL-NATIVE analytical PI
sf_native = StatsForecast(models=[AutoARIMA(season_length=M, alias="ARIMA")], freq="MS")
fc_native = sf_native.forecast(df=train, h=H, level=[95])

# (b) STATSFORECAST CONFORMAL PI: same model, split-conformal residual calibration
sf_conf = StatsForecast(models=[AutoARIMA(season_length=M, alias="ARIMA")], freq="MS")
fc_conf = sf_conf.forecast(
    df=train, h=H, level=[95],
    prediction_intervals=ConformalIntervals(n_windows=2, h=H),
)
fc_native.head(3)

## 3. CSP — training-free
`CSPModel` returns the statsforecast-style `mean / lo-95 / hi-95` dict. For a non-seasonal-safe,
horizon-adaptive interval we use `residual_mode="h_step"` (recommended for `H > m`).

In [ ]:
csp = CSPModel(season_length=M, residual_mode="h_step", orientation=True,
               n_samples=500, random_state=0)
fc_csp = csp.forecast(train["y"].to_numpy(dtype=float), h=H, level=[95])
# also keep the full predictive sample (CSP gives a distribution, not just an interval)
from csp_forecaster import ConformalSeasonalPool
csp_core = ConformalSeasonalPool(residual_mode="h_step", orientation=True, random_state=0)
csp_core.fit(train["y"].to_numpy(dtype=float), seasonal_period=M)
csp_res = csp_core.predict(H=H, alpha=0.05, n_samples=500)
print("CSP keys:", list(fc_csp.keys()))

## 4. Collect the three intervals on the common horizon

In [ ]:
def band(lo, hi, mean, name):
    return dict(name=name, mean=np.asarray(mean,float),
                lo=np.asarray(lo,float), hi=np.asarray(hi,float))

bands = [
    band(fc_native["ARIMA-lo-95"], fc_native["ARIMA-hi-95"], fc_native["ARIMA"], "ARIMA native"),
    band(fc_conf["ARIMA-lo-95"],   fc_conf["ARIMA-hi-95"],   fc_conf["ARIMA"],   "ARIMA + statsforecast conformal"),
    band(fc_csp["lo-95"],          fc_csp["hi-95"],          fc_csp["mean"],     "CSP (training-free)"),
]

## 5. Metrics: coverage, width, Winkler interval score

In [ ]:
def coverage(b):  return float(np.mean((y_true >= b["lo"]) & (y_true <= b["hi"])))
def width(b):     return float(np.mean(b["hi"] - b["lo"]))
def winkler(b, alpha=0.05):
    lo, hi = b["lo"], b["hi"]; w = hi - lo
    below = y_true < lo; above = y_true > hi
    s = w + (2/alpha)*(lo - y_true)*below + (2/alpha)*(y_true - hi)*above
    return float(np.mean(s))

rows = [dict(Method=b["name"], **{"Coverage@95%": round(coverage(b),3),
            "Mean width": round(width(b),1), "Winkler (lower=better)": round(winkler(b),1)})
        for b in bands]
summary = pd.DataFrame(rows).set_index("Method")
summary

**How to read it:** coverage should sit near **0.95**. The model-native SARIMA interval is
usually slightly under-covered on AirPassengers; the statsforecast conformal wrap re-calibrates it;
CSP reaches comparable coverage with **no training**. Among methods that hit coverage, the smaller
Winkler score is sharper-and-calibrated.

## 6. Plot the three bands

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2), sharey=True)
ctx = df.iloc[-(H+36):]
for ax, b in zip(axes, bands):
    ax.plot(ctx["ds"], ctx["y"], color="black", lw=1.2, label="actual")
    ax.plot(test["ds"], b["mean"], color="C0", lw=1.6, label="forecast")
    ax.fill_between(test["ds"], b["lo"], b["hi"], color="C0", alpha=0.25, label="95% PI")
    ax.axvline(test["ds"].iloc[0], color="grey", ls="--", lw=0.8)
    ax.set_title(f"{b['name']}\ncov={coverage(b):.2f}  width={width(b):.0f}")
    ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()

## 7. Bonus — CSP is a full distribution, not just an interval
Because CSP draws a predictive **sample** per horizon, you also get any quantile and a CRPS,
not only a 95% band.

In [ ]:
def crps_sample(samples, y):                       # samples: (H, S)
    S = samples.shape[1]; s = np.sort(samples, axis=1)
    t1 = np.mean(np.abs(s - y[:, None]), axis=1)
    t2 = np.mean(np.abs(s[:, None, :] - s[:, :, None]), axis=(1, 2))
    return float(np.mean(t1 - 0.5*t2))
print(f"CSP mean CRPS over {H} horizons: {crps_sample(csp_res.samples, y_true):.2f}")
print("CSP 50% / 80% / 95% interval widths:",
      *[f"{lv}%:{np.mean(np.ptp(np.quantile(csp_res.samples,[(1-lv/100)/2,1-(1-lv/100)/2],axis=1), axis=0)):.0f}"
        for lv in (50,80,95)])

---
### Citation
```bibtex
@misc{manokhin2026csp,
  title         = {Training-Free Probabilistic Time-Series Forecasting with Conformal Seasonal Pools},
  author        = {Manokhin, Valery},
  year          = {2026}, eprint = {2605.03789},
  archivePrefix = {arXiv}, primaryClass = {cs.LG},
  url           = {https://arxiv.org/abs/2605.03789}
}
```
Paper: <https://arxiv.org/abs/2605.03789> · Package: `pip install csp-forecaster`